# <font color="#6e8ebc">**Generación Escenario Productivo Simulado - DEMO**</font>
---

## <font color='#b0aeae'>**Índice**</font>

1. <font color="#6e8ebc">**Descripción del notebook**</font>
2. <font color="#6e8ebc">**Configuraciones**</font>
   * Importación de librerías
   * Paths
   * Paleta de Colores del Proyecto
3. <font color="#6e8ebc">**Extracción de Datos**</font>
4. <font color="#6e8ebc">**Alteración de clientes**</font>
   - Modificar Balance
   - Modificar IsActiveMember
   - Modificar NumOfProducts
5. <font color="#6e8ebc">**Generación Datos Artificiales - DEMO**</font>
   - Transacciones
   - Interacción con la App
7. <font color="#6e8ebc">**Filtrado de Datos**</font>
8. <font color="#6e8ebc">**Guardado de Datos - DEMO**</font>

# <font color="#6e8ebc">**Descripción del notebook**</font>

En el presente notebook se seleccionaron los clientes activos (Exited = 0) de la base original para simular un escenario de **'Producción Futura'**. Se realizaron alteraciones controladas en sus perfiles para inyectar patrones de riesgo (Churn Drift) que indiquen una propensión al abandono.

Adicionalmente, se generaron datos sintéticos de **transacciones y navegación en la App** proyectados al Q1 2026, replicando la lógica de generación del entrenamiento pero adaptada para reflejar comportamientos de fuga recientes (ej. drenaje de cuentas, inactividad).

Finalmente, se consolidó una muestra optimizada de **50 clientes** para garantizar la performance y reproducibilidad durante la demostración en vivo del MVP.

# <font color="#6e8ebc">**Cofiguraciones**</font>

## <font color="#ff743d">**Importación de librerías**</font>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import uuid

import os
import pickle
from pathlib import Path
from typing import Tuple, Callable, List

import warnings

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

RANDOM_SEED = 42

## <font color="#ff743d">**Paths**</font>

In [3]:
# Obtiene el path actual
PROJECT_PATH = os.getcwd()


src = Path('src')
DATA_PATH = PROJECT_PATH / src

demo = Path('demo')
DEMO_DATA_PATH = DATA_PATH / demo

champion = Path('champion')
CHAMPION_PATH = PROJECT_PATH / champion

img = Path('img')
IMG_PATH = PROJECT_PATH / img

models = Path('models')
MODELS_PATH = PROJECT_PATH / models

reports = Path('reports')
REPORTS_PATH = PROJECT_PATH / reports

dirs = [DATA_PATH, DEMO_DATA_PATH, CHAMPION_PATH, IMG_PATH, MODELS_PATH, REPORTS_PATH]


for directory in dirs:
    if not directory.exists():
        directory.mkdir(parents=True)
        print(f'Directorio creado:\n{directory}\n')
    else:
        print(f'El directorio ya existe:\n{directory}\n')

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\src

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\src\demo

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\champion

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\img

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\models

El directorio ya existe:
C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\reports



## <font color="#ff743d">**Paleta de Colores del Proyecto**</font>

**Naranjas**

<span style="display:inline-block; width:20px; height:20px; background:#993c00; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#cc5a00; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#ff7b00; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#ff9c40; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#ffbf80; margin-right:5px;"></span>

**Rojos**

<span style="display:inline-block; width:20px; height:20px; background:#7a1c00; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#b23200; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#e74903; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#ff743d; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#ffa68c; margin-right:5px;"></span>

**Azules**

<span style="display:inline-block; width:20px; height:20px; background:#001a37; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#002b5c; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#003873; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#315c99; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#6e8ebc; margin-right:5px;"></span>

**Celestes**

<span style="display:inline-block; width:20px; height:20px; background:#004752; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#007789; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#00b5d0; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#40cee6; margin-right:5px;"></span>
<span style="display:inline-block; width:20px; height:20px; background:#8ae6f4; margin-right:5px;"></span>

In [4]:
NARANJA_1, NARANJA_2, NARANJA_3, NARANJA_4, NARANJA_5 = '#993c00', '#cc5a00', '#ff7b00', '#ff9c40', '#ffbf80'
ROJO_1, ROJO_2, ROJO_3, ROJO_4, ROJO_5 = '#7a1c00', '#b23200', '#e74903', '#ff743d', '#ffa68c'
AZUL_1, AZUL_2, AZUL_3, AZUL_4, AZUL_5 = '#001a37', '#002b5c', '#003873', '#315c99', '#6e8ebc'
CIELO_1, CIELO_2, CIELO_3, CIELO_4, CIELO_5 = '#004752', '#007789', '#00b5d0', '#40cee6', '#8ae6f4'

# <font color="#6e8ebc">**Extracción de datos**</font>

In [5]:
df_clients = pd.read_parquet(DATA_PATH/'clients.parquet')
df_clients

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ExitDate,CustomerSegment
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2025-11-15,Cliente potencial
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,NaT,Standard
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,2025-12-28,VIP
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,NaT,Poco Valor
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,NaT,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,NaT,Cliente potencial
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,NaT,Valioso - Bajo compromiso
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1,2025-12-18,Poco Valor
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,2025-11-28,Standard


In [6]:
df_clients = df_clients[df_clients['Exited'] == 0]
df_clients

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ExitDate,CustomerSegment
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,NaT,Standard
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,NaT,Poco Valor
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,NaT,Standard
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0,NaT,Standard
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0,NaT,VIP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9993,9994,15569266,Rahman,644,France,Male,28,7,155060.41,1,1,0,29179.52,0,NaT,Poco Valor
9994,9995,15719294,Wood,800,France,Female,29,2,0.00,2,0,0,167773.55,0,NaT,Cliente potencial
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,NaT,Cliente potencial
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,NaT,Valioso - Bajo compromiso


In [7]:
cids = df_clients['CustomerId'].unique()
will_exit = df_clients.sample(frac=0.05, random_state=RANDOM_SEED)
will_exit

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ExitDate,CustomerSegment
8165,8166,15813503,Pickering,606,Spain,Male,37,8,154712.58,2,1,0,89099.18,0,NaT,Valioso - Bajo compromiso
6409,6410,15690695,Flynn,683,France,Female,33,9,0.00,2,1,1,38784.42,0,NaT,Cliente potencial
2157,2158,15750649,Uwakwe,744,France,Female,44,3,0.00,2,1,1,189016.14,0,NaT,Valioso - Bajo compromiso
1618,1619,15662955,Nicholls,697,France,Male,27,8,141223.68,2,1,0,90591.15,0,NaT,Standard
1633,1634,15755868,Farmer,562,France,Male,35,7,0.00,1,0,0,48869.67,0,NaT,Poco Valor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,418,15695632,Dellucci,556,France,Female,39,9,89588.35,1,1,1,94898.10,0,NaT,Valioso - Bajo compromiso
7612,7613,15599535,Howell,678,Spain,Male,28,5,138668.18,1,1,1,54144.01,0,NaT,Cliente potencial
3823,3824,15585823,Wilson,627,France,Male,31,8,128131.73,1,1,0,96131.47,0,NaT,Standard
5079,5080,15692406,Gow,427,France,Male,37,5,0.00,2,1,1,121485.10,0,NaT,Standard


In [8]:
will_exit_cids = will_exit['CustomerId']

In [9]:
df_clients_man = df_clients.copy()

In [10]:
df_clients_man.drop(['Exited', 'ExitDate'], axis=1, inplace=True)

In [11]:
df_clients_man['Will Exit'] = df_clients_man['CustomerId'].isin(will_exit_cids).astype(int)

In [12]:
df_clients_man['Will Exit'].sum()

398

In [13]:
start_date = '2026-01-01'
end_date = '2026-03-31'

In [14]:
fechas = pd.date_range(start=start_date, end=end_date, freq="D")

In [15]:
fechas_abandono = pd.date_range(start='2026-02-01', end=end_date, freq="D")

In [16]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

abandono_cust = []

for will_exit in df_clients_man['Will Exit']:
    if will_exit == 1:
        if random.random() < 0.3:
            # 30% de churners en los últimos 45 días
            rand_idx = int(np.random.triangular(0, len(fechas_abandono)-1, len(fechas_abandono)-1))
            abandono_cust.append(fechas_abandono[rand_idx])
        else:
            # 70 % churners en cualquier fecha del período
            abandono_cust.append(random.choice(fechas))
    else:
        abandono_cust.append(np.nan)


df_clients_man['Expected Date'] = abandono_cust

In [17]:
df_clients_man.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7963 entries, 1 to 9999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   RowNumber        7963 non-null   int64         
 1   CustomerId       7963 non-null   int64         
 2   Surname          7963 non-null   object        
 3   CreditScore      7963 non-null   int64         
 4   Geography        7963 non-null   object        
 5   Gender           7963 non-null   object        
 6   Age              7963 non-null   int64         
 7   Tenure           7963 non-null   int64         
 8   Balance          7963 non-null   float64       
 9   NumOfProducts    7963 non-null   int64         
 10  HasCrCard        7963 non-null   int64         
 11  IsActiveMember   7963 non-null   int64         
 12  EstimatedSalary  7963 non-null   float64       
 13  CustomerSegment  7963 non-null   object        
 14  Will Exit        7963 non-null   int32       

# <font color="#6e8ebc">**Alteración de Clientes**</font>

## <font color="#ff743d">**Modificar Balances**</font>

In [18]:
np.random.seed(RANDOM_SEED)

num_rows = len(df_clients_man)
noise_factor = 0.4
random_noise = np.random.uniform(low=-noise_factor, high=noise_factor, size=num_rows)
multiplicative_factor = 1 + random_noise

# máscara booleana de who will exit (True/False) — tiene la longitud correcta
will_exit_mask = df_clients_man['CustomerId'].isin(will_exit_cids).to_numpy()

# posiciones (0..num_rows-1) de los churners
will_exit_positions = np.nonzero(will_exit_mask)[0]

# número a forzar positivo (70%)
num_positive = int(0.6 * len(will_exit_positions))

# por si acaso: evitar pedir más de lo que hay
if num_positive > 0:
    positive_positions = np.random.choice(will_exit_positions, size=num_positive, replace=False)

    # asegurar aumento: hacer multiplicative_factor > 1 en esas posiciones
    multiplicative_factor[positive_positions] = 1 + np.abs(random_noise[positive_positions])

# aplicar
df_clients_man['Balance'] = df_clients_man['Balance'].to_numpy() * multiplicative_factor
df_clients_man['Balance'] = df_clients_man['Balance'].clip(lower=0)
df_clients_man['Balance'] = np.round(df_clients_man['Balance'], 2)
# chequeo rápido
positive_share = (multiplicative_factor[will_exit_positions] > 1).mean()

## <font color="#ff743d">**Modificar IsActiveMember**</font>

Cambiar de estado de Miembro Activo (IsActiveMember) para algunos clientes que "se irán (Will Exit)"

In [19]:
np.random.seed(RANDOM_SEED)  

MODIFY_PROPORTION = 0.3

# Máscara de los clientes que están en will_exit
will_exit_mask = df_clients_man['CustomerId'].isin(will_exit_cids)

will_exit_indices = df_clients_man.loc[will_exit_mask].index

num_to_deactivate = int(MODIFY_PROPORTION * len(will_exit_indices))

if num_to_deactivate > 0:
    deactivate_indices = np.random.choice(will_exit_indices, size=num_to_deactivate, replace=False)

    df_clients_man.loc[deactivate_indices, 'IsActiveMember'] = 0

## <font color="#ff743d">**Modificar NumOfProducts**</font>

Cambiar Cantidad de Productos (NumOfProducts) para algunos clientes que "se irán (Will Exit)"

In [20]:
np.random.seed(RANDOM_SEED) 

MODIFY_PROPORTION = 0.4

# Máscara de los clientes que están en will_exit
will_exit_mask = df_clients_man['CustomerId'].isin(will_exit_cids)

will_exit_indices = df_clients_man.loc[will_exit_mask].index

num_to_deactivate = int(MODIFY_PROPORTION * len(will_exit_indices))

if num_to_deactivate > 0:
    deactivate_indices = np.random.choice(will_exit_indices, size=num_to_deactivate, replace=False)

    df_clients_man.loc[deactivate_indices, 'NumOfProducts'] = 3

# <font color="#6e8ebc">**Generación de Datos Artificiales - DEMO**</font>

In [21]:
# --- 1. CONFIGURACIÓN DEL PERIODO DEMO ---
# Demo Day: 2026-01-27
# Generamos hasta fin de mes: 2026-01-31
# Hacia atrás 275 días (aprox 9 meses)
end_date_demo = '2026-01-31'
start_date_demo = (pd.to_datetime(end_date_demo) - pd.Timedelta(days=275)).strftime('%Y-%m-%d')

print(f"📅 Generando datos para Demo desde {start_date_demo} hasta {end_date_demo}")

# --- 2. ADAPTACIÓN DEL DATAFRAME ---
# Creamos una copia base para la simulación que renombre las columnas
# para que encajen con la lógica original de los scripts (Exited, ExitDate)
df_demo_base = df_clients_man.copy()

# Renombrar para compatibilidad con scripts existentes
df_demo_base.rename(columns={
    'Will Exit': 'Exited',
    'Expected Date': 'ExitDate'
}, inplace=True)

# Asegurar tipos de datos
df_demo_base['ExitDate'] = pd.to_datetime(df_demo_base['ExitDate'])
df_demo_base['Tenure'] = df_demo_base['Tenure'].astype(float)

# Llenar nulos de ExitDate con una fecha futura lejana para los que no se van
# (Para evitar errores en comparaciones de fecha)
df_demo_base['ExitDate'] = df_demo_base['ExitDate'].fillna(pd.to_datetime('2099-12-31'))

print(f"👥 Base de clientes lista: {len(df_demo_base)} clientes.")
print(f"   Churners inyectados (Will Exit): {df_demo_base['Exited'].sum()}")

📅 Generando datos para Demo desde 2025-05-01 hasta 2026-01-31
👥 Base de clientes lista: 7963 clientes.
   Churners inyectados (Will Exit): 398


## <font color="#ff743d">**Transacciones**</font>

In [22]:
# --- CONFIGURACIÓN Y SEMILLA ---
RNG = np.random.default_rng(RANDOM_SEED)

# --- HIPERPARÁMETROS GENERALES ---
TX_VOLUME_NOISE = 0.15   

# --- HIPERPARÁMETROS DE COMPORTAMIENTO ---
# 1. COMPORTAMIENTO DE DRENAJE (Signal - Churners)
PROB_CHURNER_DRAINS = 0.60        
DRAIN_WINDOW_DAYS = 60            
DRAIN_MULTIPLIER_MAX = 3.0        

# 2. RUIDO DE GASTO (Noise - Non-Churners)
PROB_LOYAL_BIG_SPENDER = 0.15     
LOYAL_SPIKE_MULTIPLIER = 2.5      

# 3. RUIDO DE INACTIVIDAD/PAUSAS
PROB_PAUSE_LOYAL = 0.15           
PAUSE_MIN_MONTHS = 1              
PAUSE_MAX_MONTHS = 3              

# Tipos y Multiplicadores
TX_TYPES = ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']
TX_TYPE_MULTIPLIERS = {'PAYMENT': 0.3, 'TRANSFER': 1.0, 'CASH_OUT': 0.5, 'DEBIT': 0.15, 'CASH_IN': 1.0}

# --- PARÁMETROS POR CLUSTER ---
tx_cluster_params = {
    "Poco Valor": {"base_tx_per_month": 3, "salary_ratio": 0.12, "amount_volatility": 0.2, "probs_type": [0.40, 0.10, 0.10, 0.30, 0.10]},
    "Cliente potencial": {"base_tx_per_month": 6, "salary_ratio": 0.25, "amount_volatility": 0.45, "probs_type": [0.35, 0.20, 0.10, 0.25, 0.10]},
    "Standard": {"base_tx_per_month": 10, "salary_ratio": 0.15, "amount_volatility": 0.25, "probs_type": [0.30, 0.30, 0.10, 0.20, 0.10]},
    "Valioso - Bajo compromiso": {"base_tx_per_month": 8, "salary_ratio": 0.20, "amount_volatility": 0.30, "probs_type": [0.20, 0.40, 0.20, 0.10, 0.10]},
    "VIP": {"base_tx_per_month": 20, "salary_ratio": 0.4, "amount_volatility": 0.65, "probs_type": [0.15, 0.50, 0.15, 0.05, 0.15]}
}

# --- PARÁMETROS DE VOLUMEN (DECAY) ---
LAST_N_MONTHS_DECAY = 4
DECAY_FINAL_FRAC_TX = 0.10
PROB_CHURN_REPENTINO_TX = 0.20

# --- PREPARACIÓN DEL GRID ---
freq = "M"
months = pd.period_range(start=start_date_demo, end=end_date_demo, freq=freq).to_timestamp('M')

# Usamos df_demo_base que ya tiene Exited y ExitDate mapeados correctamente
clients_tx = df_demo_base[['CustomerId', 'CustomerSegment', 'EstimatedSalary', 'Exited', 'ExitDate', 'Tenure']].copy()
mean_salary = clients_tx['EstimatedSalary'].mean()
clients_tx['EstimatedSalary'] = clients_tx['EstimatedSalary'].fillna(mean_salary)

# Grid Cliente x Mes
cust_ids = clients_tx['CustomerId'].values
n_cust = len(cust_ids)
n_months = len(months)

grid_tx = pd.DataFrame({
    'CustomerId': np.repeat(cust_ids, n_months),
    'month': np.tile(months.to_numpy(), n_cust)
})
grid_tx = grid_tx.merge(clients_tx, on='CustomerId', how='left')

# Mapa de churners repentinos
churners_repentinos_tx = (RNG.random(len(clients_tx[clients_tx['Exited'] == 1])) < PROB_CHURN_REPENTINO_TX)
client_repentino_map_tx = clients_tx[clients_tx['Exited'] == 1].set_index('CustomerId').assign(is_repentino=churners_repentinos_tx)['is_repentino'].to_dict()

# --- GENERACIÓN DE PAUSAS ---
non_churners_ids = clients_tx[clients_tx['Exited'] == 0]['CustomerId'].values
ids_with_pause = RNG.choice(non_churners_ids, size=int(len(non_churners_ids) * PROB_PAUSE_LOYAL), replace=False)

pause_rows = []
total_months_idx = len(months)

for cid in ids_with_pause:
    duration = RNG.integers(PAUSE_MIN_MONTHS, PAUSE_MAX_MONTHS + 1)
    if total_months_idx > duration:
        start_idx = RNG.integers(0, total_months_idx - duration)
        pause_start = months[start_idx]
        pause_end = months[start_idx + duration - 1]
        pause_rows.append({'CustomerId': cid, 'pause_start': pause_start, 'pause_end': pause_end})

if pause_rows:
    df_pauses = pd.DataFrame(pause_rows)
    grid_tx = grid_tx.merge(df_pauses, on='CustomerId', how='left')
else:
    grid_tx['pause_start'] = pd.NaT
    grid_tx['pause_end'] = pd.NaT

# --- CÁLCULO DE VOLUMEN (LAMBDA) ---
def compute_tx_lambda(row):
    params = tx_cluster_params.get(row['CustomerSegment'], tx_cluster_params['Standard'])
    base_lam = params['base_tx_per_month']
    
    tenure_factor = (row['Tenure'] / 60.0) * 0.2 
    mean_lam = base_lam * (1 + tenure_factor)
    noise = np.clip(RNG.normal(loc=1.0, scale=TX_VOLUME_NOISE), 0.7, 1.3)
    lam = max(0.1, mean_lam * noise)
    
    # A) Pausas
    if row['Exited'] == 0:
        if pd.notna(row['pause_start']):
            if row['pause_start'] <= row['month'] <= row['pause_end']:
                return 0.0
    
    # B) Decay Churners
    if row['Exited'] == 1:
        exit_date = pd.to_datetime(row['ExitDate'])
        # Si la fecha de salida es lejana (ej. 2099), tratamos como activo en la ventana
        if exit_date.year > 2030: 
             return lam

        month_end = row['month'] + pd.offsets.MonthEnd(0)
        
        if month_end >= exit_date:
            return 0.0
            
        is_repentino = client_repentino_map_tx.get(row['CustomerId'], False)
        
        if not is_repentino:
            diff = (exit_date.to_period('M') - row['month'].to_period('M')).n
            if diff <= LAST_N_MONTHS_DECAY and diff > 0:
                frac = diff / float(LAST_N_MONTHS_DECAY)
                decay_factor = (DECAY_FINAL_FRAC_TX + (1 - DECAY_FINAL_FRAC_TX) * frac)
                lam = lam * decay_factor
                
    return lam

print("Generando volumen de transacciones (Demo)...")
grid_tx['lambda_tx'] = grid_tx.apply(compute_tx_lambda, axis=1)
grid_tx['n_tx'] = RNG.poisson(grid_tx['lambda_tx'].values)
grid_tx = grid_tx[grid_tx['n_tx'] > 0].reset_index(drop=True)

print("Expandiendo a nivel transacción...")
rep_counts = grid_tx['n_tx'].values
total_txs = rep_counts.sum()

tx_customer = np.repeat(grid_tx['CustomerId'].values, rep_counts)
tx_month = np.repeat(grid_tx['month'].values.astype('datetime64[ns]'), rep_counts)
tx_cluster_arr = np.repeat(grid_tx['CustomerSegment'].values, rep_counts)
tx_salary_arr = np.repeat(grid_tx['EstimatedSalary'].values, rep_counts)

# --- ASIGNACIÓN DE MONTOS ---
print("Asignando montos base...")
tx_types = np.empty(total_txs, dtype=object)
tx_amounts = np.zeros(total_txs, dtype=float)

for clust_name, params in tx_cluster_params.items():
    mask = (tx_cluster_arr == clust_name)
    n_cluster = mask.sum()
    if n_cluster == 0: continue
    
    types_chosen = RNG.choice(TX_TYPES, size=n_cluster, p=params['probs_type'])
    tx_types[mask] = types_chosen
    
    salaries = tx_salary_arr[mask]
    monthly_income = salaries / 12.0
    base_target = monthly_income * params['salary_ratio']
    type_factors = np.vectorize(TX_TYPE_MULTIPLIERS.get)(types_chosen)
    target_amounts = base_target * type_factors
    
    volatility = params['amount_volatility']
    noise = np.clip(RNG.normal(loc=1.0, scale=volatility, size=n_cluster), 0.5, 2.0)
    
    final_amounts = np.clip(target_amounts * noise, 5.0, salaries / 3.0)
    tx_amounts[mask] = np.round(final_amounts, 2)

month_starts = pd.to_datetime(tx_month).to_period('M').to_timestamp()
month_ends = (pd.to_datetime(tx_month).to_period('M') + 1).to_timestamp() - pd.Timedelta(days=1)
days_in_month = (month_ends - month_starts).days + 1
rand_days = RNG.integers(0, days_in_month, size=total_txs)
tx_dates = month_starts + pd.to_timedelta(rand_days, unit='D')

# --- DATAFRAME FINAL Y DRENAJE ---
df_synth_demo = pd.DataFrame({
    'CustomerId': tx_customer,
    'TransactionDate': tx_dates,
    'Amount': tx_amounts,
    'TransactionType': tx_types,
    'EstimatedSalary': tx_salary_arr
})

df_synth_demo = df_synth_demo.merge(clients_tx[['CustomerId', 'Exited', 'ExitDate']], on='CustomerId', how='left')
df_synth_demo['ExitDate'] = pd.to_datetime(df_synth_demo['ExitDate'])

print("\n🔥 Aplicando lógica de 'Drenaje' (Demo)...")
# A) SEÑAL
churner_ids = clients_tx[clients_tx['Exited'] == 1]['CustomerId'].values
if len(churner_ids) > 0:
    draining_churners = RNG.choice(churner_ids, size=int(len(churner_ids) * PROB_CHURNER_DRAINS), replace=False)
    
    df_synth_demo['days_to_exit'] = (df_synth_demo['ExitDate'] - df_synth_demo['TransactionDate']).dt.days
    
    mask_drain_signal = (
        (df_synth_demo['CustomerId'].isin(draining_churners)) & 
        (df_synth_demo['TransactionType'] == 'CASH_OUT') & 
        (df_synth_demo['days_to_exit'] >= 0) & 
        (df_synth_demo['days_to_exit'] <= DRAIN_WINDOW_DAYS)
    )
    
    factor_urgencia = 1 - (df_synth_demo.loc[mask_drain_signal, 'days_to_exit'] / DRAIN_WINDOW_DAYS)
    multipliers_signal = 1.0 + (factor_urgencia * (DRAIN_MULTIPLIER_MAX - 1.0))
    df_synth_demo.loc[mask_drain_signal, 'Amount'] *= multipliers_signal

# B) RUIDO
loyal_ids = clients_tx[clients_tx['Exited'] == 0]['CustomerId'].values
if len(loyal_ids) > 0:
    big_spenders_loyal = RNG.choice(loyal_ids, size=int(len(loyal_ids) * PROB_LOYAL_BIG_SPENDER), replace=False)
    mask_potential_noise = (
        (df_synth_demo['CustomerId'].isin(big_spenders_loyal)) & 
        (df_synth_demo['TransactionType'] == 'CASH_OUT')
    )
    if mask_potential_noise.sum() > 0:
        noise_indices = df_synth_demo[mask_potential_noise].sample(frac=0.10, random_state=RANDOM_SEED).index
        df_synth_demo.loc[noise_indices, 'Amount'] *= LOYAL_SPIKE_MULTIPLIER

# Limpieza
df_synth_demo['Amount'] = np.clip(df_synth_demo['Amount'], 0, df_synth_demo['EstimatedSalary']).round(2)
df_synth_demo['TransactionId'] = np.arange(len(df_synth_demo)).astype(str)
df_transactions_demo = df_synth_demo[['TransactionId', 'CustomerId', 'TransactionDate', 'Amount', 'TransactionType']].sort_values(['CustomerId', 'TransactionDate']).reset_index(drop=True)

print(f"\n--- Transacciones Demo Generadas: {len(df_transactions_demo)} ---")

Generando volumen de transacciones (Demo)...
Expandiendo a nivel transacción...
Asignando montos base...

🔥 Aplicando lógica de 'Drenaje' (Demo)...

--- Transacciones Demo Generadas: 558456 ---


## <font color="#ff743d">**Interacción con la App**</font>

In [23]:
# --- CONFIGURACIÓN Y SEMILLA ---
RNG = np.random.default_rng(RANDOM_SEED)

# Parámetros de Ruido
LAMBDA_NOISE_SIGMA = 0.20 
DECAY_NOISE_SIGMA = 0.25   
PROB_NOISE_SIGMA = 0.15    

# --- PERIODOS Y CLUSTERS ---
freq = "M"
months = pd.period_range(start=start_date_demo, end=end_date_demo, freq=freq).to_timestamp('M')

# Parámetros por cluster (copiados de tu script)
cluster_params = {
    "Poco Valor": {"base_sessions_per_month": 1.5, "duration_mu": 2.0, "duration_sigma": 0.6, "p_transfer": 0.02, "p_payment": 0.05, "p_invest": 0.01, "p_push_open": 0.10, "p_failed_login": 0.01},
    "Cliente potencial": {"base_sessions_per_month": 3.0, "duration_mu": 2.3, "duration_sigma": 0.7, "p_transfer": 0.06, "p_payment": 0.08, "p_invest": 0.02, "p_push_open": 0.15, "p_failed_login": 0.02},
    "Standard": {"base_sessions_per_month": 4.0, "duration_mu": 2.7, "duration_sigma": 0.8, "p_transfer": 0.10, "p_payment": 0.12, "p_invest": 0.03, "p_push_open": 0.18, "p_failed_login": 0.03},
    "Valioso - Bajo compromiso": {"base_sessions_per_month": 3.5, "duration_mu": 2.6, "duration_sigma": 0.8, "p_transfer": 0.12, "p_payment": 0.10, "p_invest": 0.05, "p_push_open": 0.12, "p_failed_login": 0.02},
    "VIP": {"base_sessions_per_month": 8.0, "duration_mu": 3.0, "duration_sigma": 0.9, "p_transfer": 0.25, "p_payment": 0.20, "p_invest": 0.15, "p_push_open": 0.35, "p_failed_login": 0.01}
}

# Parámetros Churn/Pausa
prob_churn_repentino_app = 0.20 
last_n_months_decay = 6       
decay_final_frac = 0.05       
PROB_PAUSA_NO_CHURNER_APP = 0.15 
PAUSA_DURACION_MIN_MESES = 2       
PAUSA_DURACION_MAX_MESES = 4       

# --- PREPARACIÓN DEL GRID (Usando df_demo_base) ---
clients = df_demo_base[['CustomerId','CustomerSegment','Exited','ExitDate','Tenure']].copy()
tenure_max = clients['Tenure'].max()
tenure_weight = 0.5

cust_ids = clients['CustomerId'].values
n_cust = len(cust_ids)
n_months = len(months)
grid = pd.DataFrame({
    'CustomerId': np.repeat(cust_ids, n_months),
    'month': np.tile(months.to_numpy(), n_cust)
})
grid = grid.merge(clients, on='CustomerId', how='left')

churners_repentinos = (RNG.random(len(clients[clients['Exited'] == 1])) < prob_churn_repentino_app)
client_repentino_map = clients[clients['Exited'] == 1].set_index('CustomerId').assign(is_repentino=churners_repentinos)['is_repentino'].to_dict()

# --- PAUSAS DE ACTIVIDAD ---
no_churners = clients[clients['Exited'] == 0].copy()
if len(no_churners) > 0:
    clientes_con_pausa_ids = no_churners.sample(frac=PROB_PAUSA_NO_CHURNER_APP, random_state=RANDOM_SEED)['CustomerId']
    pause_info = []
    total_meses = len(months)
    for cid in clientes_con_pausa_ids:
        duracion_pausa = np.random.randint(PAUSA_DURACION_MIN_MESES, PAUSA_DURACION_MAX_MESES + 1)
        if total_meses > duracion_pausa:
            start_offset = np.random.randint(0, total_meses - duracion_pausa)
            pausa_inicio = months[start_offset]
            pausa_fin = months[start_offset + duracion_pausa - 1]
            pause_info.append({'CustomerId': cid, 'pause_start': pausa_inicio, 'pause_end': pausa_fin})
    
    if pause_info:
        df_pausas = pd.DataFrame(pause_info)
        grid = grid.merge(df_pausas, on='CustomerId', how='left')
    else:
        grid['pause_start'] = pd.NaT; grid['pause_end'] = pd.NaT
else:
    grid['pause_start'] = pd.NaT; grid['pause_end'] = pd.NaT

grid['pause_start'] = grid['pause_start'].fillna(pd.NaT)
grid['pause_end'] = grid['pause_end'].fillna(pd.NaT)

# --- FUNCIÓN LAMBDA ---
def compute_lambda_row(row):
    # Mapping para fallback si el segmento no está exacto
    params = cluster_params.get(row['CustomerSegment'], cluster_params.get("Standard", cluster_params["Poco Valor"]))
    
    base_lam = params['base_sessions_per_month']
    tenure_factor = (row['Tenure'] / tenure_max) * tenure_weight if tenure_max > 0 else 0
    mean_lam = base_lam * (1 + tenure_factor)
    noise_factor_base = np.clip(RNG.normal(loc=1.0, scale=LAMBDA_NOISE_SIGMA), 0.5, 2.0)
    lam = max(0.01, mean_lam * noise_factor_base)

    if row['Exited'] == 0:
        is_in_pause = pd.notna(row['pause_start']) and (row['month'] >= row['pause_start']) and (row['month'] <= row['pause_end'])
        return 0.0 if is_in_pause else lam
    else:
        exit_date = pd.to_datetime(row['ExitDate'])
        if exit_date.year > 2030: return lam # Safety net

        month_end = row['month'] + pd.offsets.MonthEnd(0)
        if month_end >= exit_date: return 0.0
        
        is_repentino = client_repentino_map.get(row['CustomerId'], False)
        if is_repentino: return lam
        else:
            months_before = int(((exit_date.to_period('M') - row['month'].to_period('M')).n))
            if months_before <= last_n_months_decay and months_before > 0:
                frac = months_before / float(last_n_months_decay)
                base_decay_factor = (decay_final_frac + (1 - decay_final_frac) * frac)
                noise_decay_factor = np.clip(RNG.normal(loc=1.0, scale=DECAY_NOISE_SIGMA), 0.5, 1.5)
                lam = lam * base_decay_factor * noise_decay_factor
                return max(0.01, lam)
    return lam

print("Generando Sesiones Demo...")
grid['lambda'] = grid.apply(compute_lambda_row, axis=1)
grid['n_sessions'] = RNG.poisson(grid['lambda'].values)
grid = grid[grid['n_sessions'] > 0].reset_index(drop=True)

# --- EXPANSIÓN A SESIONES ---
rep_counts = grid['n_sessions'].values
total_sessions = rep_counts.sum()
session_customer = np.repeat(grid['CustomerId'].values, rep_counts)
session_month = np.repeat(grid['month'].values.astype('datetime64[ns]'), rep_counts)

cust_to_cluster = clients.set_index('CustomerId')['CustomerSegment'].to_dict()
session_cluster = np.vectorize(lambda c: cust_to_cluster.get(c, "Standard"))(session_customer)

mu_map = {k: v['duration_mu'] for k, v in cluster_params.items()}
sigma_map = {k: v['duration_sigma'] for k, v in cluster_params.items()}
mu_arr = np.vectorize(lambda c: mu_map.get(c, 2.7))(session_cluster)
sigma_arr = np.vectorize(lambda c: sigma_map.get(c, 0.8))(session_cluster)
durations = np.clip(RNG.lognormal(mean=mu_arr, sigma=sigma_arr), 0.5, 180.0)

# Probabilidades y eventos
def get_param_vec(param_name, clusters_arr):
    # Fallback a "Standard" si el cluster no existe
    return np.vectorize(lambda cl: cluster_params.get(cl, cluster_params['Standard'])[param_name])(clusters_arr)

p_transfer = np.clip(get_param_vec('p_transfer', session_cluster) * np.clip(RNG.normal(1.0, PROB_NOISE_SIGMA, total_sessions), 0.5, 1.5), 0.001, 0.999)
p_payment  = np.clip(get_param_vec('p_payment', session_cluster) * np.clip(RNG.normal(1.0, PROB_NOISE_SIGMA, total_sessions), 0.5, 1.5), 0.001, 0.999)
p_invest   = np.clip(get_param_vec('p_invest', session_cluster) * np.clip(RNG.normal(1.0, PROB_NOISE_SIGMA, total_sessions), 0.5, 1.5), 0.001, 0.999)
p_push     = np.clip(get_param_vec('p_push_open', session_cluster) * np.clip(RNG.normal(1.0, PROB_NOISE_SIGMA, total_sessions), 0.5, 1.5), 0.001, 0.999)
p_fail     = np.clip(get_param_vec('p_failed_login', session_cluster) * np.clip(RNG.normal(1.0, PROB_NOISE_SIGMA, total_sessions), 0.5, 1.5), 0.001, 0.999)

used_transfer = RNG.random(total_sessions) < p_transfer
used_payment  = RNG.random(total_sessions) < p_payment
used_invest   = RNG.random(total_sessions) < p_invest
opened_push   = RNG.random(total_sessions) < p_push
failed_login  = RNG.random(total_sessions) < p_fail

month_starts = pd.to_datetime(session_month).to_period('M').to_timestamp()
month_ends = (pd.to_datetime(session_month).to_period('M') + 1).to_timestamp() - pd.Timedelta(days=1)
days_in_month = (month_ends - month_starts).days + 1
rand_days = RNG.integers(0, days_in_month, size=total_sessions)
session_dates = month_starts + pd.to_timedelta(rand_days, unit='D')

df_sessions_demo = pd.DataFrame({
    'SessionId': np.arange(total_sessions).astype(str),
    'CustomerId': session_customer,
    'SessionDate': session_dates,
    'DurationMin': durations,
    'UsedTransfer': used_transfer.astype(int),
    'UsedPayment': used_payment.astype(int),
    'UsedInvest': used_invest.astype(int),
    'OpenedPush': opened_push.astype(int),
    'FailedLogin': failed_login.astype(int)
})

print(f"\n--- Sesiones Demo Generadas: {len(df_sessions_demo):,} ---")

Generando Sesiones Demo...

--- Sesiones Demo Generadas: 295,155 ---


In [24]:
df_sessions_demo

,SessionId,CustomerId,SessionDate,DurationMin,UsedTransfer,UsedPayment,UsedInvest,OpenedPush,FailedLogin
0,0,15647311,2025-05-11,29.832016,1,1,0,1,1
1,1,15647311,2025-06-10,10.610990,0,0,0,0,0
2,2,15647311,2025-06-13,14.285189,0,0,0,0,0
3,3,15647311,2025-07-07,35.034859,0,0,0,0,0
4,4,15647311,2025-07-08,8.636829,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...
295150,295150,15628319,2025-11-30,8.538733,0,0,0,0,0
295151,295151,15628319,2025-11-04,4.250089,0,0,0,0,0
295152,295152,15628319,2025-12-27,7.818867,0,0,0,0,0
295153,295153,15628319,2026-01-21,10.602499,0,0,0,0,0


# <font color="#6e8ebc">**Filtrado de Datos**</font>

In [25]:
df_clients_man_NoChurn = df_clients_man[df_clients_man['Will Exit'] == 0].sample(n=40, random_state=RANDOM_SEED)
df_clients_man_Churn = df_clients_man[df_clients_man['Will Exit'] == 1].sample(n=10, random_state=RANDOM_SEED)

In [26]:
df_demo_50 = pd.concat([df_clients_man_NoChurn, df_clients_man_Churn])
df_demo_50 = df_demo_50.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

In [27]:
df_demo_50

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,CustomerSegment,Will Exit,Expected Date
0,7746,15701166,Chinedum,660,France,Male,40,5,133046.41,2,1,1,38761.61,Valioso - Bajo compromiso,0,NaT
1,1582,15576517,Everingham,445,Germany,Female,34,7,144318.36,2,1,1,70618.00,Valioso - Bajo compromiso,0,NaT
2,2953,15654901,Horton,733,France,Male,51,10,114107.05,1,1,0,130189.53,Valioso - Bajo compromiso,0,NaT
3,1892,15574783,Perkins,584,France,Female,37,1,0.00,3,1,0,180363.56,Standard,1,2026-01-20
4,3951,15602841,Lockett,794,Spain,Female,28,5,0.00,2,0,1,86699.98,Poco Valor,0,NaT
5,5568,15787222,Ch'in,676,Germany,Male,28,1,79700.88,3,1,0,128461.29,Standard,1,2026-03-07
6,4540,15574206,Shillito,718,France,Female,37,7,0.00,2,1,1,55100.09,Standard,0,NaT
7,5208,15751022,Bowhay,777,Germany,Female,37,10,140766.61,2,1,1,73464.88,VIP,0,NaT
8,8091,15601324,Black,697,France,Female,48,1,0.00,2,1,1,87400.53,Valioso - Bajo compromiso,0,NaT
9,2973,15592877,Wright,641,Spain,Male,42,9,150547.50,1,1,0,35367.19,Standard,0,NaT


In [28]:
# 1. Filtrar correctamente SOLO los IDs de los Churners (Will Exit = 1)
churner_ids = df_demo_50[df_demo_50['Will Exit'] == 1]['CustomerId']

# 2. Guardar como JSON
churner_ids.to_json(DEMO_DATA_PATH / 'will_exit_ids.json', orient='values')

print(f"✅ Se guardaron {len(churner_ids)} IDs de 'Will Exit' para pruebas.")

✅ Se guardaron 10 IDs de 'Will Exit' para pruebas.


In [29]:
# 1. Filtrar correctamente SOLO los IDs de los Churners (Will Exit = 1)
no_churner_ids = df_demo_50[df_demo_50['Will Exit'] == 0]['CustomerId']

# 2. Guardar como JSON
no_churner_ids.to_json(DEMO_DATA_PATH / 'will_not_exit_ids.json', orient='values')

print(f"✅ Se guardaron {len(no_churner_ids)} IDs de 'Will Exit' para pruebas.")

✅ Se guardaron 40 IDs de 'Will Exit' para pruebas.


In [30]:
df_demo_50 = df_demo_50.drop(['CustomerSegment', 'Will Exit', 'Expected Date'], axis=1)

In [31]:
# Selccionar clientes IDS de clientes para el sample
target_ids = df_demo_50['CustomerId'].unique()

print(f"🎯 Filtrando datos para {len(target_ids)} clientes seleccionados...")

# Filtrar Transacciones
# Usamos .isin() para mantener solo las filas de nuestros clientes VIP de la demo
df_transactions_50 = df_transactions_demo[df_transactions_demo['CustomerId'].isin(target_ids)].copy()
df_transactions_50 = df_transactions_50.reset_index(drop=True)

# Filtrar Sesiones de App
df_sessions_50 = df_sessions_demo[df_sessions_demo['CustomerId'].isin(target_ids)].copy()
df_sessions_50 = df_sessions_50.reset_index(drop=True)

# --- VERIFICACIÓN DE INTEGRIDAD (CRÍTICO PARA LA DEMO) ---
# Verificamos que no hayamos perdido clientes en el camino y que el volumen sea manejable

print("\n📊 RESUMEN FINAL PARA BASE DE DATOS (LIGERA):")
print("-" * 50)

print(f"👥 Clientes Maestros:    {len(df_demo_50)}")

print(f"💳 Transacciones:        {len(df_transactions_50):,} filas")
print(f"   - Clientes únicos:    {df_transactions_50['CustomerId'].nunique()} (Debe ser <= 50)")
print(f"   - Fecha min:          {df_transactions_50['TransactionDate'].min()}")
print(f"   - Fecha max:          {df_transactions_50['TransactionDate'].max()}")

print(f"📱 Sesiones App:         {len(df_sessions_50):,} filas")
print(f"   - Clientes únicos:    {df_sessions_50['CustomerId'].nunique()} (Debe ser <= 50)")
print(f"   - Fecha min:          {df_sessions_50['SessionDate'].min()}")
print(f"   - Fecha max:          {df_sessions_50['SessionDate'].max()}")

print("-" * 50)

# Alerta si algún cliente se quedó sin datos
clientes_sin_tx = set(target_ids) - set(df_transactions_50['CustomerId'].unique())
if clientes_sin_tx:
    print(f"⚠️ ATENCIÓN: Hay {len(clientes_sin_tx)} clientes sin transacciones generadas.")
else:
    print("✅ Todos los clientes tienen historial transaccional.")

clientes_sin_ss = set(target_ids) - set(df_sessions_50['CustomerId'].unique())
if clientes_sin_ss:
    print(f"⚠️ ATENCIÓN: Hay {len(clientes_sin_ss)} clientes sin sesiones generadas.")
else:
    print("✅ Todos los clientes tienen historial de interacciones con la aplicación.")

🎯 Filtrando datos para 50 clientes seleccionados...

📊 RESUMEN FINAL PARA BASE DE DATOS (LIGERA):
--------------------------------------------------
👥 Clientes Maestros:    50
💳 Transacciones:        3,951 filas
   - Clientes únicos:    50 (Debe ser <= 50)
   - Fecha min:          2025-05-01 00:00:00
   - Fecha max:          2026-01-31 00:00:00
📱 Sesiones App:         1,541 filas
   - Clientes únicos:    50 (Debe ser <= 50)
   - Fecha min:          2025-05-01 00:00:00
   - Fecha max:          2026-01-31 00:00:00
--------------------------------------------------
✅ Todos los clientes tienen historial transaccional.
✅ Todos los clientes tienen historial de interacciones con la aplicación.


In [32]:
df_demo_50.loc[df_demo_50['CustomerId'] == 15736534, 'IsActiveMember'] = 1
df_demo_50

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,7746,15701166,Chinedum,660,France,Male,40,5,133046.41,2,1,1,38761.61
1,1582,15576517,Everingham,445,Germany,Female,34,7,144318.36,2,1,1,70618.00
2,2953,15654901,Horton,733,France,Male,51,10,114107.05,1,1,0,130189.53
3,1892,15574783,Perkins,584,France,Female,37,1,0.00,3,1,0,180363.56
4,3951,15602841,Lockett,794,Spain,Female,28,5,0.00,2,0,1,86699.98
5,5568,15787222,Ch'in,676,Germany,Male,28,1,79700.88,3,1,0,128461.29
6,4540,15574206,Shillito,718,France,Female,37,7,0.00,2,1,1,55100.09
7,5208,15751022,Bowhay,777,Germany,Female,37,10,140766.61,2,1,1,73464.88
8,8091,15601324,Black,697,France,Female,48,1,0.00,2,1,1,87400.53
9,2973,15592877,Wright,641,Spain,Male,42,9,150547.50,1,1,0,35367.19


# <font color="#6e8ebc">**Guardado de Datos - DEMO**</font>

In [33]:
import json

# --- CONFIGURACIÓN DE EXPORTACIÓN ---
# Usamos 'records' para obtener una lista de objetos: [{col:val}, {col:val}...]
# Usamos 'iso' para las fechas: "2026-01-27T..." (El estándar web)

print("💾 Guardando archivos JSON para la Demo...")

# 1. Clientes (Master Data)
# Guardamos en archivo
df_demo_50.to_json(
    DEMO_DATA_PATH / 'demo_clients.json', 
    orient='records', 
    date_format='iso', 
    indent=4 # Indentación para que sea legible
)
print("   ✅ demo_clients.json guardado.")


# 2. Transacciones (Transactional Data)
df_transactions_50.to_json(
    DEMO_DATA_PATH / 'demo_transactions.json', 
    orient='records', 
    date_format='iso', 
    indent=4
)
print("   ✅ demo_transactions.json guardado.")


# 3. Sesiones de App (Behavioral Data)
df_sessions_50.to_json(
    DEMO_DATA_PATH / 'demo_sessions.json', 
    orient='records', 
    date_format='iso', 
    indent=4
)
print("   ✅ demo_sessions.json guardado.")


# --- VISTA PREVIA (Para que verifiques el formato) ---
print("\n🔍 VISTA PREVIA (Primer registro de Cliente):")
# Leemos el string JSON generado para mostrarte cómo se ve
json_preview = df_demo_50.head(1).to_json(orient='records', date_format='iso')
parsed = json.loads(json_preview)
print(json.dumps(parsed, indent=4))

💾 Guardando archivos JSON para la Demo...
   ✅ demo_clients.json guardado.
   ✅ demo_transactions.json guardado.
   ✅ demo_sessions.json guardado.

🔍 VISTA PREVIA (Primer registro de Cliente):
[
    {
        "RowNumber": 7746,
        "CustomerId": 15701166,
        "Surname": "Chinedum",
        "CreditScore": 660,
        "Geography": "France",
        "Gender": "Male",
        "Age": 40,
        "Tenure": 5,
        "Balance": 133046.41,
        "NumOfProducts": 2,
        "HasCrCard": 1,
        "IsActiveMember": 1,
        "EstimatedSalary": 38761.61
    }
]


In [34]:
import random
import pandas as pd
from datetime import datetime

# --- CONFIGURACIÓN ---
OUTPUT_SQL_FILE = DEMO_DATA_PATH / 'insert_demo_data.sql'
TODAY = datetime.now()
TODAY = pd.Timestamp(TODAY) 


# Productos estáticos
PRODUCTS_LIST = [
    (1, 'Credit Card'), (2, 'Savings Account'), (3, 'Personal Loan'),
    (4, 'Mortgage'), (5, 'Investment Fund'), (6, 'Insurance')
]

def escape_sql(val):
    if isinstance(val, str):
        return val.replace("'", "''")
    return val

print("🛠️ Generando Script SQL con Lógica de IDs Secuenciales...")

# --- PASO CRÍTICO: GENERAR EL MAPA DE IDs (Real -> Secuencial) ---
# Como df_demo_50 ya está ordenado/mezclado como queremos,
# simplemente enumeramos del 1 al 50.
customer_map = {} # { Real_ID : Secuencial_ID }
for idx, row in df_demo_50.reset_index(drop=True).iterrows():
    sequential_id = idx + 1 # 1, 2, 3...
    real_id = row['CustomerId']
    customer_map[real_id] = sequential_id

print(f"   Mapping generado para {len(customer_map)} clientes.")

with open(OUTPUT_SQL_FILE, 'w', encoding='utf-8') as f:
    
    f.write("-- SCRIPT GENERADO PARA DEMO CUSTECH (IDS SECUENCIALES)\n")
    f.write(f"-- FECHA: {pd.Timestamp.now()}\n\n")

    # 3. PRODUCTS
    f.write("-- 3. PRODUCTS\n")
    f.write("INSERT IGNORE INTO product (id, name) VALUES\n")
    prods_sql = [f"({pid}, '{name}')" for pid, name in PRODUCTS_LIST]
    f.write(",\n".join(prods_sql) + ";\n\n")

    # 1. CUSTOMERS 
    # (AQUÍ USAMOS EL ID REAL, PORQUE ASÍ LO PIDE EL INSERT)
    f.write("-- 1. CUSTOMERS\n")
    f.write("INSERT IGNORE INTO customer (customer_id, geography, gender, surname, birth_date, estimated_salary, created_at) VALUES\n")
    
    values_cust = []
    # Iteramos el DF original para mantener el orden exacto del 1 al 50
    for _, row in df_demo_50.reset_index(drop=True).iterrows():
        birth_date = (TODAY - pd.DateOffset(years=int(row['Age']))).date()
        created_at = (TODAY - pd.DateOffset(years=int(row['Tenure']))).date()
        
        # NOTA: Aquí insertamos row['CustomerId'] (el 15xxxx) como valor explícito
        val = f"({row['CustomerId']}, '{row['Geography']}', '{row['Gender']}', '{escape_sql(row['Surname'])}', '{birth_date}', {row['EstimatedSalary']:.2f}, '{created_at}')"
        values_cust.append(val)
    f.write(",\n".join(values_cust) + ";\n\n")

    # 2. CUSTOMER STATUS 
    # (AQUÍ USAMOS EL ID SECUENCIAL)
    f.write("-- 2. CUSTOMER STATUS\n")
    f.write("INSERT IGNORE INTO customer_status (customer_id, credit_score, is_active_member, has_cr_card) VALUES\n")
    
    values_status = []
    for _, row in df_demo_50.iterrows():
        seq_id = customer_map[row['CustomerId']] # Traducimos al ID 1, 2, 3...
        val = f"({seq_id}, {row['CreditScore']}, {row['IsActiveMember']}, {row['HasCrCard']})"
        values_status.append(val)
    f.write(",\n".join(values_status) + ";\n\n")

    # 4. ACCOUNTS 
    # (AQUÍ USAMOS EL ID SECUENCIAL)
    f.write("-- 4. ACCOUNTS\n")
    f.write("INSERT IGNORE INTO account (customer_id, balance, opened_at, closed_at) VALUES\n")
    
    values_acc = []
    for _, row in df_demo_50.iterrows():
        seq_id = customer_map[row['CustomerId']] # Traducción
        opened_at = (TODAY - pd.DateOffset(years=int(row['Tenure']))).date()
        closed_at = "NULL" 
        val = f"({seq_id}, {row['Balance']:.2f}, '{opened_at}', {closed_at})"
        values_acc.append(val)
    f.write(",\n".join(values_acc) + ";\n\n")

    # 5. CUSTOMER_PRODUCT 
    # (AQUÍ USAMOS EL ID SECUENCIAL)
    f.write("-- 5. CUSTOMER_PRODUCT\n")
    f.write("INSERT IGNORE INTO customer_product (customer_id, product_id) VALUES\n")
    
    values_cp = []
    product_ids_avail = [p[0] for p in PRODUCTS_LIST]
    
    for _, row in df_demo_50.iterrows():
        seq_id = customer_map[row['CustomerId']] # Traducción
        num_prods = int(row['NumOfProducts'])
        if num_prods > 0:
            assigned_prods = random.sample(product_ids_avail, k=min(num_prods, len(product_ids_avail)))
            for pid in assigned_prods:
                values_cp.append(f"({seq_id}, {pid})")
    f.write(",\n".join(values_cp) + ";\n\n")

    # 6. TRANSACTIONS 
    # (AQUÍ USAMOS EL ID SECUENCIAL)
    f.write("-- 6. TRANSACTIONS\n")
    f.write("INSERT IGNORE INTO customer_transaction (transaction_id, transaction_date, amount, transaction_type, customer_id) VALUES\n")
    
    values_tx = []
    for _, row in df_transactions_50.iterrows():
        seq_id = customer_map[row['CustomerId']] # Traducción usando el mapa
        
        t_date = row['TransactionDate'].strftime('%Y-%m-%d %H:%M:%S')
        t_id = f"TXN-{row['TransactionId']}"
        
        val = f"('{t_id}', '{t_date}', {row['Amount']:.2f}, '{row['TransactionType']}', {seq_id})"
        values_tx.append(val)
    f.write(",\n".join(values_tx) + ";\n\n")

    # 7. SESSIONS 
    # (AQUÍ USAMOS EL ID SECUENCIAL)
    f.write("-- 7. SESSIONS\n")
    f.write("INSERT IGNORE INTO customer_session (session_id, session_date, duration_min, used_transfer, used_payment, used_invest, opened_push, failed_login, customer_id) VALUES\n")
    
    values_sess = []
    for _, row in df_sessions_50.iterrows():
        seq_id = customer_map[row['CustomerId']] # Traducción usando el mapa
        
        s_date = row['SessionDate'].strftime('%Y-%m-%d %H:%M:%S')
        s_id = f"SESS-{row['SessionId']}"
        
        val = f"('{s_id}', '{s_date}', {row['DurationMin']:.2f}, {row['UsedTransfer']}, {row['UsedPayment']}, {row['UsedInvest']}, {row['OpenedPush']}, {row['FailedLogin']}, {seq_id})"
        values_sess.append(val)
    f.write(",\n".join(values_sess) + ";\n")

print(f"✅ SQL Generado. Revisar: {OUTPUT_SQL_FILE}")

🛠️ Generando Script SQL con Lógica de IDs Secuenciales...
   Mapping generado para 50 clientes.
✅ SQL Generado. Revisar: C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\src\demo\insert_demo_data.sql


In [35]:
import json
import pandas as pd

# --- CONFIGURACIÓN DE SALIDA ---
BATCH_FILE = DEMO_DATA_PATH / 'api_batch_payload.json'
CHURNERS_DIR = DEMO_DATA_PATH / 'churners_json'

# Crear directorio para los individuales si no existe
CHURNERS_DIR.mkdir(parents=True, exist_ok=True)

print("🚀 Generando JSONs para pruebas de API...")

# 1. FUNCIÓN HELPER: CONVERTIR DATAFRAMES A ESTRUCTURA ANIDADA
def build_api_payload(df_client, df_tx, df_ss):
    payloads = []
    
    # Iteramos por cada cliente del DataFrame maestro
    for _, row_cli in df_client.iterrows():
        cid = row_cli['CustomerId']
        
        # A. Construir objeto Cliente
        # Nos aseguramos de convertir tipos de numpy a nativos de python (int, float) para que JSON no falle
        cliente_data = {
            "CustomerId": str(cid), # La API espera str
            "Surname": str(row_cli['Surname']),
            "CreditScore": int(row_cli['CreditScore']),
            "Geography": str(row_cli['Geography']),
            "Gender": str(row_cli['Gender']),
            "Age": int(row_cli['Age']),
            "Tenure": int(row_cli['Tenure']),
            "Balance": float(row_cli['Balance']),
            "NumOfProducts": int(row_cli['NumOfProducts']),
            "HasCrCard": int(row_cli['HasCrCard']),
            "IsActiveMember": int(row_cli['IsActiveMember']),
            "EstimatedSalary": float(row_cli['EstimatedSalary'])
        }
        
        # B. Buscar y formatear Transacciones
        # Filtramos las transacciones de este cliente
        tx_cli = df_tx[df_tx['CustomerId'] == cid]
        transacciones_data = []
        for _, row_tx in tx_cli.iterrows():
            tx_obj = {
                "TransactionId": str(row_tx['TransactionId']),
                # Formato ISO YYYY-MM-DD es seguro
                "TransactionDate": row_tx['TransactionDate'].strftime('%Y-%m-%d'),
                "Amount": float(row_tx['Amount']),
                "TransactionType": str(row_tx['TransactionType'])
            }
            transacciones_data.append(tx_obj)
            
        # C. Buscar y formatear Sesiones
        # SessionId, SessionDate, DurationMin, FailedLogin
        ss_cli = df_ss[df_ss['CustomerId'] == cid]
        sesiones_data = []
        for _, row_ss in ss_cli.iterrows():
            ss_obj = {
                "SessionId": str(row_ss['SessionId']),
                "SessionDate": row_ss['SessionDate'].strftime('%Y-%m-%d'),
                "DurationMin": float(row_ss['DurationMin']),
                "FailedLogin": int(row_ss['FailedLogin'])
            }
            sesiones_data.append(ss_obj)
            
        # D. Ensamblar objeto completo (FullCustomerData)
        full_customer = {
            "cliente": cliente_data,
            "transacciones": transacciones_data,
            "sesiones": sesiones_data
        }
        
        payloads.append(full_customer)
        
    return payloads

# --- 2. GENERAR EL BATCH COMPLETO (50 CLIENTES) ---
full_batch_payload = build_api_payload(df_demo_50, df_transactions_50, df_sessions_50)

# Guardar Batch
with open(BATCH_FILE, 'w', encoding='utf-8') as f:
    json.dump(full_batch_payload, f, indent=4)

print(f"✅ Batch generado: {BATCH_FILE}")
print(f"   -> Contiene {len(full_batch_payload)} clientes.")


# --- 3. GENERAR JSONs INDIVIDUALES PARA LOS CHURNERS ---
# Identificamos a los que 'Will Exit' == 1 en tu dataframe original de la demo
# (Asegúrate de tener esa columna o usa los IDs que guardamos antes)

# Opción A: Si df_demo_50 todavía tiene la columna 'Will Exit' (o 'Exited')
# churner_rows = df_demo_50[df_demo_50['Exited'] == 1] 

# Opción B: Usando la lista de IDs que guardamos en 'will_exit_ids.json'
# (Asumiendo que cargaste esa lista o usas la lógica de filtro directo)
churner_ids = df_demo_50[df_demo_50['CustomerId'].isin(churner_ids)]['CustomerId'].values
# Nota: Puse apellidos de tu ejemplo anterior que parecían churners, 
# pero mejor usa el filtro directo si tienes la columna 'Exited' o 'Will Exit'.
# Si ya borraste la columna, usa los IDs que filtramos para el SQL.

# Vamos a asumir que puedes filtrar el subset de churners:
# (Aquí filtro los primeros 10 del batch que tengan transacciones recientes o sean interesantes)
# Para tu caso, tomaremos los 10 primeros del batch que coincidan con tus churners.
churners_payload = [p for p in full_batch_payload if p['cliente']['CustomerId'] in [str(x) for x in churner_ids]] 

# Si la lista anterior está vacía (porque no tengo tus IDs exactos en memoria), 
# tomo los últimos 10 del batch (que solían ser los churners en tu lógica de concat):
if not churners_payload:
    churners_payload = full_batch_payload[-10:] 

print(f"🎯 Generando {len(churners_payload)} archivos individuales para Churners...")

for i, data in enumerate(churners_payload):
    cid = data['cliente']['CustomerId']
    surname = data['cliente']['Surname']
    filename = CHURNERS_DIR / f"churner_{i+1}_{surname}_{cid}.json"
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)
        
print(f"✅ Archivos individuales guardados en: {CHURNERS_DIR}")

🚀 Generando JSONs para pruebas de API...
✅ Batch generado: C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\src\demo\api_batch_payload.json
   -> Contiene 50 clientes.
🎯 Generando 10 archivos individuales para Churners...
✅ Archivos individuales guardados en: C:\Users\Ignacio\OneDrive\Desktop\hackathon_ONE\churn-insight-ml\src\demo\churners_json
